<a href="https://colab.research.google.com/github/Ayush-Singh-36/stock_prediction_pytorch/blob/secondary/stock_pricing_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Development

# Device-agnostic code

In [28]:
import torch
from torch import cuda
device = "cuda" if cuda.is_available else "cpu"
print(device)

cuda


# Importing dataset from kaggle

In [29]:
import os
from google.colab import userdata
import sys
def custom_exit(status):
    print(f"Kaggle API tried to exit with status {status}. Ignoring for Colab environment.")
    # Optionally, raise an exception or log, instead of actual exit.
sys.exit = custom_exit
exit = custom_exit # Patch the global 'exit' function as well
try:
    __builtins__.exit = custom_exit # Explicitly patch built-in exit
except AttributeError:
    print("Could not patch __builtins__.exit - it might not be present or modifiable in this environment.")

# Retrieve credentials from Colab secrets
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Now import and run the download code
from kaggle.api.kaggle_api_extended import KaggleApi

dataset_slug = "emrekaany/google-daily-stock-prices-2004-today"
download_path = "./data"

print("Authenticating via environment variables...")
api = KaggleApi()
api.authenticate()

print("Downloading movie dataset from Kaggle...")
api.dataset_download_files(dataset_slug, path=download_path, unzip=True)

print(f"Done! Your files have been saved to the '{download_path}' folder.")

Authenticating via environment variables...
Dataset URL: https://www.kaggle.com/datasets/emrekaany/google-daily-stock-prices-2004-today
Done! Your files have been saved to the './data' folder.


# Checking for cardinality of data

In [30]:
import pandas as pd
import numpy as np
def profile_dataset_features(df: pd.DataFrame, max_categories_to_print: int = 10):
    """
    Scans a massive dataset to automatically break down columns into
    categorical option spaces or numerical statistical boundaries.
    """
    print(f"=== Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns ===\n")

    # 1. Separate column types automatically
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns

    print(f"Found {len(categorical_cols)} Categorical columns and {len(numerical_cols)} Numerical columns.\n")
    print("-" * 50)
    print("CATEGORICAL FEATURE OPTIONS DISCOVERY")
    print("-" * 50)

    # 2. Extract Options from Categorical Columns
    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        num_unique = len(unique_vals)
        missing_count = df[col].isna().sum()

        print(f"\n🔹 Feature: '{col}' | Unique Values Count: {num_unique} | Missing: {missing_count} rows")

        # High cardinality warning (e.g., IDs, Hash keys, open text fields)
        if num_unique > 30:
            print(f"  ⚠️ High Cardinality! Showing first 5 options sample: {list(unique_vals[:5])}...")
        else:
            # Print value distributions so you know if an option is incredibly rare
            value_counts = df[col].value_counts(dropna=False)
            for val, count in value_counts.items():
                pct = (count / len(df)) * 100
                print(f"  - [{val}]: {count} occurrences ({pct:.2f}%)")

    print("\n" + "-" * 50)
    print("NUMERICAL FEATURE BOUNDARY DISCOVERY")
    print("-" * 50)

    # 3. Extract Ranges from Numerical Columns
    # Using describe gives you min, max, and percentiles to spot extreme values or outliers
    numeric_summary = df[numerical_cols].describe().T[['min', 'max', 'mean']]
    display(numeric_summary)

# --- Example of running it on your data ---
data = pd.read_csv("./data/googl_daily_prices.csv")
profile_dataset_features(data)

=== Dataset Shape: 5350 rows | 6 columns ===

Found 1 Categorical columns and 5 Numerical columns.

--------------------------------------------------
CATEGORICAL FEATURE OPTIONS DISCOVERY
--------------------------------------------------

🔹 Feature: 'date' | Unique Values Count: 5350 | Missing: 0 rows
  ⚠️ High Cardinality! Showing first 5 options sample: ['2025-11-20', '2025-11-19', '2025-11-18', '2025-11-17', '2025-11-14']...

--------------------------------------------------
NUMERICAL FEATURE BOUNDARY DISCOVERY
--------------------------------------------------


,min,max,mean
1. open,85.40,3.025000e+03,7.521667e+02
2. high,86.52,3.030932e+03,7.596804e+02
3. low,83.34,2.977980e+03,7.442795e+02
4. close,83.43,2.996770e+03,7.521181e+02
5. volume,465638.00,1.277476e+08,1.035159e+07


# Transfrormation

In [31]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, StandardScaler

numerical_cols = data.select_dtypes(include = [np.number]).columns
categorical_cols = data.select_dtypes(include = ['object']).columns

if 'date' in categorical_cols:
    categorical_cols = categorical_cols.drop('date')

encoder = TargetEncoder()
scaler = StandardScaler()
preprocessor = ColumnTransformer(
    transformers = [
        ('num', scaler, numerical_cols),
        ('cat', encoder, categorical_cols)
    ]
)

x_transformed = preprocessor.fit_transform(data.drop(columns = ['date']))

# Dividing data into training, validation and test dataset

In [34]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def create_sequences(data, seq_length = 30, target_col_idx = 3):
  xs, ys = [], []
  for i in range(len(data) - seq_length):
    x = data[i : (i + seq_length)]
    y = data[i + seq_length, target_col_idx]
    xs.append(x)
    ys.append(y)
  return np.array(xs), np.array(ys)

sequence_length = 30
x_sequences, y_sequences = create_sequences(x_transformed, seq_length = sequence_length)

total_len = len(x_sequences)
train_size = int(total_len * 0.70)
val_size = int(total_len * 0.15)

x_train, y_train = x_sequences[:train_size], y_sequences[:train_size]
x_val, y_val = (
    x_sequences[train_size : train_size + val_size],
    y_sequences[train_size : train_size + val_size]
)
x_test, y_test = x_sequences[train_size + val_size:], y_sequences[train_size + val_size :]

train_dataset = TensorDataset(
    torch.tensor(x_train, dtype = torch.float32),
    torch.tensor(y_train, dtype = torch.float32).unsqueeze(1)
)
val_dataset = TensorDataset(
    torch.tensor(x_val, dtype = torch.float32),
    torch.tensor(y_val, dtype = torch.float32).unsqueeze(1)
)
test_dataset = TensorDataset(
    torch.tensor(x_test, dtype = torch.float32),
    torch.tensor(y_test, dtype = torch.float32).unsqueeze(1)
)

train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)

# Model Architecture

In [35]:
import torch
import torch.nn as nn

class rnn_lstm_model(nn.Module):
  def __init__(self, input_size, rnn_hidden_size, lstm_hidden_size, output_size, num_layers = 1):
    super(rnn_lstm_model, self).__init__()

    self.rnn = nn.RNN(
        input_size = input_size,
        hidden_size = rnn_hidden_size,
        num_layers = num_layers,
        batch_first = True
    )

    self.lstm = nn.LSTM(
        input_size = rnn_hidden_size,
        hidden_size = lstm_hidden_size,
        num_layers = num_layers,
        batch_first = True
    )

    self.fc = nn.Linear(lstm_hidden_size, output_size)

  def forward(self, x):
    rnn_out, _ = self.rnn(x)
    lstm_out, (h_n, c_n) = self.lstm(rnn_out)
    last_time_step_out = lstm_out[:, -1, :]
    out = self.fc(last_time_step_out)
    return out

input_features = 5
sequence_length = 30
batch_size = 32

model = rnn_lstm_model(
    input_size = input_features,
    rnn_hidden_size = 64,
    lstm_hidden_size = 128,
    output_size = 1
).to(device)

# Loss fn & Optimizer

In [36]:
import torch.nn as nn
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

In [37]:
epochs = 50

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_x.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss * 100:.6f}")

Epoch [1/50] - Loss: 9.551411
Epoch [5/50] - Loss: 0.571803
Epoch [10/50] - Loss: 0.515639
Epoch [15/50] - Loss: 0.458737
Epoch [20/50] - Loss: 0.480253
Epoch [25/50] - Loss: 0.481418
Epoch [30/50] - Loss: 0.527912
Epoch [35/50] - Loss: 0.440416
Epoch [40/50] - Loss: 0.478504
Epoch [45/50] - Loss: 0.452775
Epoch [50/50] - Loss: 0.440621
